# TP1 — Collecte et exploration des données
**Projet fil rouge : Cyberbullying Detection (texte + image)**

Pipeline (scripts in `src/`, run from the project root):
1. `python src/reddit_scraper.py` → `data/reddit_raw.csv` + `data/images/`
2. `python src/cleaning.py` → `data/cyberbullying_raw.csv`, `data/cyberbullying_raw_clean.csv`
3. This notebook → Part 4 analysis + mini-report, logged to MLflow

> ⚠️ The corpus contains offensive language (it is the object of study).

In [ ]:
import json, re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# works whether the notebook is launched from the root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA, REPORTS = ROOT / "data", ROOT / "reports"
REPORTS.mkdir(exist_ok=True)

clean = pd.read_csv(DATA / "cyberbullying_raw_clean.csv")
raw = pd.read_csv(DATA / "cyberbullying_raw.csv")

# bring back subreddit / query_group from the raw metadata (joined on id)
meta = raw.set_index("id")["metadata"].map(json.loads)
clean["subreddit"] = clean["id"].map(meta.map(lambda m: m["subreddit"]))
clean["query_group"] = clean["id"].map(meta.map(lambda m: m["query_group"]))
clean["media"] = clean["media"].fillna("")
clean["n_chars"] = clean["clean_text"].fillna("").str.len()
clean["n_words"] = clean["clean_text"].fillna("").str.split().str.len()
clean.head(3)

## Partie 4.1 — Analyse statistique minimale

In [ ]:
stats = {
    "n_messages": len(clean),
    "n_raw": len(raw),
    "n_images": int((clean["media"] != "").sum()),
    "image_rate": (clean["media"] != "").mean(),
    "mean_chars": clean["n_chars"].mean(),
    "mean_words": clean["n_words"].mean(),
    "median_words": clean["n_words"].median(),
}
for k, v in stats.items():
    print(f"{k:<14} {v:,.2f}" if isinstance(v, float) else f"{k:<14} {v:,}")

print("\nRépartition par source :")
print(clean["source"].value_counts().to_string())
print("\nRépartition par subreddit :")
print(clean["subreddit"].value_counts().to_string())

In [ ]:
BLUE, INK, MUTED = "#2a78d6", "#0b0b0b", "#52514e"

def style(ax, title):
    ax.set_title(title, loc="left", color=INK, fontsize=12)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(colors=MUTED)
    ax.grid(axis="x", color="#e6e5e0", linewidth=0.8)
    ax.set_axisbelow(True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

counts = clean["subreddit"].value_counts().sort_values()
axes[0].barh(counts.index, counts.values, color=BLUE, height=0.6)
for y, v in enumerate(counts.values):
    axes[0].text(v + 1, y, str(v), va="center", color=MUTED, fontsize=9)
style(axes[0], "Messages par subreddit")

axes[1].hist(clean["n_words"].clip(upper=300), bins=40, color=BLUE, rwidth=0.9)
axes[1].axvline(stats["mean_words"], color=INK, linestyle="--", linewidth=1)
axes[1].text(stats["mean_words"] + 4, axes[1].get_ylim()[1] * 0.9,
             f"moyenne = {stats['mean_words']:.0f} mots", color=INK, fontsize=9)
style(axes[1], "Longueur des textes (mots, coupé à 300)")
axes[1].grid(axis="y", color="#e6e5e0", linewidth=0.8); axes[1].grid(axis="x", visible=False)

plt.tight_layout()
fig.savefig(REPORTS / "tp1_distribution.png", dpi=120)
plt.show()

## Partie 4.2 — Analyse qualitative
### 10 exemples variés (2 par groupe de requête)
`query_group` = *comment le message a été trouvé* (mot-clé), **pas** son étiquette.

In [ ]:
examples = (clean[clean["query_group"] != "image_post"]
            .groupby("query_group")
            .sample(2, random_state=42))
for _, r in examples.iterrows():
    print(f"[{r.query_group:<7} | r/{r.subreddit}] {r.clean_text[:200]}\n")

### Diversité lexicale

In [ ]:
STOP = set(ENGLISH_STOP_WORDS) | {"just", "like", "don", "people", "really", "think", "know"}
tokens = [w for t in clean["clean_text"].fillna("")
          for w in re.findall(r"[a-z']+", t) if len(w) > 1]
freq = Counter(tokens)
vocab = len(freq)
hapax = sum(1 for c in freq.values() if c == 1)

lex = {
    "n_tokens": len(tokens),
    "vocab_size": vocab,
    "ttr": vocab / len(tokens),                 # type/token ratio
    "hapax_ratio": hapax / vocab,               # words seen only once
}
for k, v in lex.items():
    print(f"{k:<12} {v:,.3f}" if isinstance(v, float) else f"{k:<12} {v:,}")

top = [(w, c) for w, c in freq.most_common(300) if w not in STOP][:20]
print("\nTop 20 mots (hors stopwords) :", ", ".join(f"{w} ({c})" for w, c in top))

### Pertinence du vocabulaire par rapport au thème
Part des messages contenant au moins un mot du lexique « harcèlement ».
On s'attend à un taux élevé pour les groupes toxiques et faible pour `neutral`.

In [ ]:
THEME = re.compile(
    r"\b(bull(y|ied|ying)|harass\w*|hate|kill|loser|ugly|stupid|idiot|threat\w*|"
    r"toxic|disgusting|pathetic|fat|worthless|kys|subhuman|cope|seethe|retard\w*|"
    r"moron|dumb|trash|shut up)\b")

clean["theme_hit"] = clean["clean_text"].fillna("").str.contains(THEME)
relevance = clean.groupby("query_group")["theme_hit"].mean().sort_values(ascending=False)
print(relevance.map("{:.0%}".format).to_string())
print(f"\nGlobal : {clean['theme_hit'].mean():.0%}")

## Mini-rapport (10 lignes max)

In [ ]:
import sys
sys.path.insert(0, str(ROOT / "src"))
from cleaning import SEXUAL_WORDS          # same lexicon as the cleaning step

report = f"""# Mini-rapport TP1 — Collecte
1. **Source** : Reddit via l'archive Arctic Shift (API Reddit fermée aux nouvelles clés depuis le 31/10/2026). X exclu (API payante, snscrape cassé) ; 4chan exclu (robots.txt : `Disallow: /`) mais présent indirectement via r/4chan et r/greentext.
2. **Subreddits** : {clean['subreddit'].nunique()} ({', '.join(sorted(clean['subreddit'].unique()))}) ; fenêtre 01/05–01/11/2026.
3. **Mots-clés** : insult (ugly, loser, stupid, idiot…), threat (kill yourself, kys…), hate (subhuman, go back to…), irony (cope, seethe, ratio…) + échantillon neutre sans mot-clé.
4. **Taille** : {stats['n_raw']} messages bruts → {stats['n_messages']} après nettoyage, dont {stats['n_images']} avec image ({stats['image_rate']:.1%}) ; longueur moyenne {stats['mean_words']:.0f} mots.
5. **Diversité** : {lex['vocab_size']:,} mots distincts, TTR {lex['ttr']:.3f}, {lex['hapax_ratio']:.0%} d'hapax → vocabulaire riche et varié.
6. **Pertinence** : {clean['theme_hit'].mean():.0%} des messages contiennent un mot du lexique du harcèlement (vs {relevance.get('neutral', 0):.0%} pour le groupe neutre).
7. **Bruit** : messages de bots/modérateurs, marqueurs [removed] et doublons supprimés ; la recherche plein-texte (stemming) ramène des faux positifs (« find you » → « finding my purpose »).
8. **Biais** : le groupe *threat* contient surtout des victimes racontant le harcèlement, et la classe *hate* est sous-représentée (Reddit modère fortement).
9. **Contenu sexuel** : plafonné à 20 % ; seulement {clean['clean_text'].fillna('').str.contains(SEXUAL_WORDS).mean():.1%} après filtrage NSFW à la collecte.
10. **Limite** : une seule source (Reddit) — choix à valider avec l'enseignante.
"""
(ROOT / "mini_report.md").write_text(report, encoding="utf-8")
print(report)

## Partie 6 — Traçabilité MLflow

In [ ]:
import mlflow

mlflow.set_tracking_uri(f"sqlite:///{ROOT / 'mlflow.db'}")
mlflow.set_experiment("TP1_Collecte")
with mlflow.start_run(run_name="analysis"):
    mlflow.log_metrics({**{k: float(v) for k, v in stats.items()},
                        **{k: float(v) for k, v in lex.items()},
                        "theme_relevance": float(clean["theme_hit"].mean())})
    mlflow.log_artifact(str(ROOT / "mini_report.md"))
    mlflow.log_artifact(str(REPORTS / "tp1_distribution.png"), "figures")
print("Run logged → open `mlflow ui` (experiment TP1_Collecte)")